In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
### 所需要的所有文件：
# 合并物流-财务-产品组-核算价-国内-用于低效-长尾 
# PLM的生命周期全表，因为涉及到停止销售时间，所以这里需要PLM的生命周期全表

In [3]:
month_date = 202512
channel = ['零售','工程','电商']
current_date = pd.Timestamp('2024-12-31')

productgroup_map={
    '吸油烟机': ['吸油烟机'],
    '灶具': ['灶具'],
    '蒸烤微合计': ['烤箱','蒸箱','微波炉','蒸烤烹饪机','蒸烤微烹饪机','蒸微'],
    '灶集成': ['灶消烹饪机','灶蒸烹饪机','灶蒸烤烹饪机','灶烤烹饪机'],
    '消毒柜': ['消毒柜'],
    '热水器': ['热水器','两用炉'],
    '净水机': ['家用净水机','商用净水机'],
    '洗碗机': ['水槽洗碗机','嵌入式洗碗机']
}
productgroup_sort_map = {
    '吸油烟机': 0,
    '灶具': 1,
    '蒸烤微合计': 2,
    '灶集成': 3,
    '消毒柜': 4,
    '热水器': 5,
    '净水机': 6,
    '洗碗机': 7
}
pro_group_type_map={
    '吸油烟机': '吸油烟机',
    '灶具': '灶具',
    '烤箱': '蒸烤微合计',
    '蒸箱': '蒸烤微合计',
    '微波炉': '蒸烤微合计',
    '蒸烤烹饪机': '蒸烤微合计',
    '蒸烤微烹饪机': '蒸烤微合计',
    '蒸微': '蒸烤微合计',
    '灶消烹饪机': '灶集成',
    '灶蒸烹饪机': '灶集成',
    '灶蒸烤烹饪机': '灶集成',
    '灶烤烹饪机': '灶集成',
    '消毒柜': '消毒柜',
    '热水器': '热水器',
    '两用炉': '热水器',
    '家用净水机': '家用净水机',
    '商用净水机': '商用净水机',
    '水槽洗碗机': '水槽/嵌入洗碗机合计',
    '嵌入式洗碗机': '水槽/嵌入洗碗机合计'
}


### 读取物流数据，只保留3大渠道、并且是国内的数据

In [4]:
df_dcs = pd.read_excel(fr"D:\000物料报表\202511\24年发货.xlsx")
# df['物料号'] = df['商品编码'].astype(str).map(lambda x: x[:13])
# print(len(df))
print(df_dcs.columns)
df_dcs


Index(['发货类别', '发货仓', '渠道', '大区/渠道A', '收货片区/门店', '收货城市', '收货仓', '是否三四级',
       '承运方式', '计划发货时间', '实际发货时间', '商品编码', '商品名称', '产品简称', '库存状态', '计划总数量',
       '实际出库数量', '产品品类', '销售产品品类', '考核产品品类', '年份', '月份', '周次', '发货月份'],
      dtype='object')


,发货类别,发货仓,渠道,大区/渠道A,收货片区/门店,收货城市,收货仓,是否三四级,承运方式,计划发货时间,...,库存状态,计划总数量,实际出库数量,产品品类,销售产品品类,考核产品品类,年份,月份,周次,发货月份
0,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,成品,3,3,欧式油烟机,油烟机,欧式油烟机,2024,1,1,2024年1月
1,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,成品,19,19,灶具,灶具,灶具,2024,1,1,2024年1月
2,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,成品,9,9,欧式油烟机,油烟机,欧式油烟机,2024,1,1,2024年1月
3,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,成品,2,2,欧式油烟机,油烟机,欧式油烟机,2024,1,1,2024年1月
4,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,成品,18,18,近吸式油烟机,油烟机,近吸式油烟机,2024,1,1,2024年1月
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
525024,物流出库单,第一工业园立体仓,海外,海外,NaN,NaN,NaN,否,海外出库,2024-12-06,...,成品,25,25,欧式油烟机,油烟机,欧式油烟机,2024,12,50,2024年12月
525025,物流出库单,第一工业园立体仓,海外,海外,NaN,NaN,NaN,否,海外出库,2024-12-06,...,成品,15,15,蒸箱,蒸烤微,微蒸烤,2024,12,50,2024年12月
525026,物流出库单,第一工业园立体仓,海外,海外,NaN,NaN,NaN,否,海外出库,2024-12-27,...,成品,1,1,欧式油烟机,油烟机,欧式油烟机,2024,12,52,2024年12月
525027,物流出库单,第一工业园立体仓,海外,海外,NaN,NaN,NaN,否,海外出库,2024-12-27,...,成品,1,1,微波炉,蒸烤微,微蒸烤,2024,12,52,2024年12月


In [5]:
df_dcs['渠道'].value_counts()


渠道
零售    475165
工程     23827
电商     23716
海外      2166
米博       149
商净         6
Name: count, dtype: int64

In [6]:
df_plm = pd.read_excel(r"D:\000物料报表\202512\单型号贡献-低效-长尾\产品生命周期状态全表.xlsx")
df_plm

e:\python\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,产品线,产品组,标准型号,是否标准型号,简化型号,国内/海外,物料号,关联项目,产品所有者,产品型号,当前评审阶段,产品状态,产品状态开始时间,下属渠道,对应渠道状态,开始销售时间,退市预警时间,停止销售时间,停止发货时间,剩余负卖生产数量
0,烹饪厨电产品线,灶具,02-CS34BW,是,02-CS34BW,国内,NaN,P-2022088-02-CS34BW,褚武建,02-CS34BW,PDCP,开发,6/8/2022 1:20:51 PM,零售,未售,NaN,NaN,NaN,NaN,0.0
1,烹饪厨电产品线,灶具,02-CS34BW,是,02-CS34BW,国内,NaN,P-2022088-02-CS34BW,褚武建,02-CS34BW,PDCP,开发,6/8/2022 1:20:51 PM,电商,未售,NaN,NaN,NaN,NaN,0.0
2,烹饪厨电产品线,灶具,02-CS34BW,是,02-CS34BW,国内,NaN,P-2022088-02-CS34BW,褚武建,02-CS34BW,PDCP,开发,6/8/2022 1:20:51 PM,工程,未售,NaN,NaN,NaN,NaN,0.0
3,洗碗机产品线,水槽洗碗机,NaN,NaN,1,国内,NaN,NaN,王伟,1,NaN,作废,11/30/2023 6:48:57 PM,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,净热产品线,热水器,JSG15-0606,NaN,JSG15-0606,国内,1004000200090,NaN,PlatformAdmin,10T-JSG15-0606FR,结项阶段,停止发货,11/30/2023 6:49:04 PM,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9382,油烟机产品线,吸油烟机,苍穹V6R1,是,苍穹V6R1,国内,NaN,P-2025272-苍穹V6R1平台开发,刘世宇,苍穹V6R1,PDCP,开发,11/24/2025 3:05:37 PM,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9383,油烟机产品线,吸油烟机,轩辕V1R1,是,轩辕V1R1,国内,NaN,P-2023108-轩辕V1R1平台,王紫军,轩辕V1R1,结项阶段,量产,11/30/2023 6:48:56 PM,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9384,油烟机产品线,吸油烟机,轩辕V1R1C02,是,轩辕V1R1C02,国内,NaN,P-2024133-轩辕V1R1C02 平台开发,徐烽,轩辕V1R1C02,结项阶段,样机,6/27/2024 9:09:52 AM,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9385,油烟机产品线,吸油烟机,轩辕V1R1C03,是,轩辕V1R1C03,国内,NaN,P-2025113-轩辕V1R1C03平台,谢靖宇,轩辕V1R1C03,PDCP,开发,6/9/2025 2:05:58 PM,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
df_dcs['物料编码'] = df_dcs['商品编码'].astype(str).str[:13]
df_plm['物料编码'] = df_plm['物料号'].astype(str).str[:13]

df_dcs['渠道'] = df_dcs['渠道'].astype(str)
df_plm['渠道'] = df_plm['下属渠道'].astype(str)

In [11]:
df = pd.merge(df_dcs, df_plm[['物料编码', '渠道', '停止销售时间','停止发货时间','产品组','标准型号','产品型号','国内/海外']], on=['物料编码', '渠道'], how='left')
df

,发货类别,发货仓,渠道,大区/渠道A,收货片区/门店,收货城市,收货仓,是否三四级,承运方式,计划发货时间,...,月份,周次,发货月份,物料编码,停止销售时间,停止发货时间,产品组,标准型号,产品型号,国内/海外
0,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,1,1,2024年1月,1001002400000,6/19/2025 9:20:56 AM,NaN,吸油烟机,J1,CXW-258-J1(不带罩),国内
1,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,1,1,2024年1月,1002003700002,1/4/2026 12:12:01 PM,NaN,灶具,02-HECB,JZT-02-HECB-12T,国内
2,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,1,1,2024年1月,1001000500376,6/19/2025 9:20:56 AM,NaN,吸油烟机,JQ31A,CXW-358-JQ32A(不带罩）,国内
3,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,1,1,2024年1月,1001000500361,8/30/2024 3:22:25 PM,12/8/2025 12:00:00 PM,吸油烟机,JQ01TY,CXW-258-JQ01TY(不带罩),国内
4,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,1,1,2024年1月,1001001500097,11/24/2025 4:51:46 PM,NaN,吸油烟机,Z7T,CXW-358-Z7T（不带罩）,国内
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
525024,物流出库单,第一工业园立体仓,海外,海外,NaN,NaN,NaN,否,海外出库,2024-12-06,...,12,50,2024年12月,1001000900312,NaN,NaN,吸油烟机,NaN,EMS9026-USFA,海外
525025,物流出库单,第一工业园立体仓,海外,海外,NaN,NaN,NaN,否,海外出库,2024-12-06,...,12,50,2024年12月,1007000400036,11/6/2025 1:34:31 PM,NaN,蒸箱,NaN,SCD42-C2T-USFA,海外
525026,物流出库单,第一工业园立体仓,海外,海外,NaN,NaN,NaN,否,海外出库,2024-12-27,...,12,52,2024年12月,1001000900379,NaN,NaN,吸油烟机,NaN,EMG9036-BDMK,海外
525027,物流出库单,第一工业园立体仓,海外,海外,NaN,NaN,NaN,否,海外出库,2024-12-27,...,12,52,2024年12月,1006000300066,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
df = df[df['渠道'].isin(['零售','工程','电商'])].reset_index(drop=True)
print(df['渠道'].value_counts())
df['停止销售时间'] = pd.to_datetime(df['停止销售时间'], format='mixed')
df['停止发货时间'] = pd.to_datetime(df['停止发货时间'], format='mixed')
df

渠道
零售    475165
工程     23827
电商     23716
Name: count, dtype: int64


,发货类别,发货仓,渠道,大区/渠道A,收货片区/门店,收货城市,收货仓,是否三四级,承运方式,计划发货时间,...,月份,周次,发货月份,物料编码,停止销售时间,停止发货时间,产品组,标准型号,产品型号,国内/海外
0,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,1,1,2024年1月,1001002400000,2025-06-19 09:20:56,NaT,吸油烟机,J1,CXW-258-J1(不带罩),国内
1,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,1,1,2024年1月,1002003700002,2026-01-04 12:12:01,NaT,灶具,02-HECB,JZT-02-HECB-12T,国内
2,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,1,1,2024年1月,1001000500376,2025-06-19 09:20:56,NaT,吸油烟机,JQ31A,CXW-358-JQ32A(不带罩）,国内
3,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,1,1,2024年1月,1001000500361,2024-08-30 15:22:25,2025-12-08 12:00:00,吸油烟机,JQ01TY,CXW-258-JQ01TY(不带罩),国内
4,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,1,1,2024年1月,1001001500097,2025-11-24 16:51:46,NaT,吸油烟机,Z7T,CXW-358-Z7T（不带罩）,国内
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522703,调拨出库单,第二工业园立体仓,工程,NaN,无,深圳市,LTC惠州RDC工程仓,NaN,干线tms,2024-12-31,...,12,1,2024年12月,1001000800336,2022-07-07 12:00:00,NaT,吸油烟机,EH37,CXW-258-EH38,国内
522704,调拨出库单,第二工业园立体仓,零售,NaN,无,南京市,华东大区事业合作商-南京库,NaN,干线tms,2024-12-31,...,12,1,2024年12月,1003000500027,NaT,NaT,消毒柜,ZTD100F-J29,ZTD100F-J29,国内
522705,调拨出库单,第二工业园立体仓,零售,NaN,无,南京市,华东大区事业合作商-南京库,NaN,干线tms,2024-12-31,...,12,1,2024年12月,1003000500019,NaT,NaT,消毒柜,ZTD100J-J25S,ZTD100J-J25S,国内
522706,调拨出库单,第二工业园立体仓,零售,NaN,无,南京市,华东大区事业合作商-南京库,NaN,干线tms,2024-12-31,...,12,1,2024年12月,1009001100002,2025-08-01 13:38:39,NaT,蒸烤烹饪机,ZK50-01-F1.i,ZK50-01-F1.i,国内


In [18]:
# 长尾只看3大渠道。每个渠道的停止销售时间和既定时间的差距
df1 = df.copy()
df1['停止发货时间'] = df1['停止发货时间'].fillna('2099-01-01')
df1 = df1[(df1['停止销售时间'].notnull())&(df1['停止发货时间']>='2025-01-01')].reset_index(drop=True)
df1 = df1[['物料编码','渠道','产品组','标准型号','国内/海外','产品型号','停止销售时间','停止发货时间']].drop_duplicates().reset_index(drop=True)
df1

,物料编码,渠道,产品组,标准型号,国内/海外,产品型号,停止销售时间,停止发货时间
0,1001002400000,零售,吸油烟机,J1,国内,CXW-258-J1(不带罩),2025-06-19 09:20:56,2099-01-01 00:00:00
1,1002003700002,零售,灶具,02-HECB,国内,JZT-02-HECB-12T,2026-01-04 12:12:01,2099-01-01 00:00:00
2,1001000500376,零售,吸油烟机,JQ31A,国内,CXW-358-JQ32A(不带罩）,2025-06-19 09:20:56,2099-01-01 00:00:00
3,1001000500361,零售,吸油烟机,JQ01TY,国内,CXW-258-JQ01TY(不带罩),2024-08-30 15:22:25,2025-12-08 12:00:00
4,1001001500097,零售,吸油烟机,Z7T,国内,CXW-358-Z7T（不带罩）,2025-11-24 16:51:46,2099-01-01 00:00:00
...,...,...,...,...,...,...,...,...
892,1001002100005,电商,吸油烟机,JCD7,国内,CXW-258-JCD7(不带罩),2023-10-20 12:00:00,2025-03-10 00:00:00
893,1003000700004,电商,消毒柜,ZTD100S-KC2.i,国内,ZTD100S-KC2.i,2025-01-06 09:41:08,2025-03-10 00:00:00
894,1002003400169,工程,灶具,TH88G,国内,JZT-TH89G-12T,2024-10-24 16:08:08,2025-09-23 12:00:00
895,1018001100001,工程,嵌入式洗碗机,JBCD7E-03-Y1.i,国内,JBCD7E-03-Y1.i,2025-09-26 10:23:29,2099-01-01 00:00:00


### 保留各个渠道停止销售的产品

In [19]:
from calendar import month
from pandas import DateOffset
current_date = pd.Timestamp('2024-12-31')
for index,row in df1.iterrows():
    if row['渠道'] == '零售':
        if row['停止销售时间'] < current_date - pd.DateOffset(months=12):
            df1.loc[index,'是否长尾型号'] = '是'
    if row['渠道'] == '电商':
        if row['停止销售时间'] < current_date - pd.DateOffset(months=12):
            df1.loc[index,'是否长尾型号'] = '是'
    if row['渠道'] == '工程':
        if row['停止销售时间'] < current_date - pd.DateOffset(months=30):
            df1.loc[index,'是否长尾型号'] = '是'
df1


,物料编码,渠道,产品组,标准型号,国内/海外,产品型号,停止销售时间,停止发货时间,是否长尾型号
0,1001002400000,零售,吸油烟机,J1,国内,CXW-258-J1(不带罩),2025-06-19 09:20:56,2099-01-01 00:00:00,NaN
1,1002003700002,零售,灶具,02-HECB,国内,JZT-02-HECB-12T,2026-01-04 12:12:01,2099-01-01 00:00:00,NaN
2,1001000500376,零售,吸油烟机,JQ31A,国内,CXW-358-JQ32A(不带罩）,2025-06-19 09:20:56,2099-01-01 00:00:00,NaN
3,1001000500361,零售,吸油烟机,JQ01TY,国内,CXW-258-JQ01TY(不带罩),2024-08-30 15:22:25,2025-12-08 12:00:00,NaN
4,1001001500097,零售,吸油烟机,Z7T,国内,CXW-358-Z7T（不带罩）,2025-11-24 16:51:46,2099-01-01 00:00:00,NaN
...,...,...,...,...,...,...,...,...,...
892,1001002100005,电商,吸油烟机,JCD7,国内,CXW-258-JCD7(不带罩),2023-10-20 12:00:00,2025-03-10 00:00:00,是
893,1003000700004,电商,消毒柜,ZTD100S-KC2.i,国内,ZTD100S-KC2.i,2025-01-06 09:41:08,2025-03-10 00:00:00,NaN
894,1002003400169,工程,灶具,TH88G,国内,JZT-TH89G-12T,2024-10-24 16:08:08,2025-09-23 12:00:00,NaN
895,1018001100001,工程,嵌入式洗碗机,JBCD7E-03-Y1.i,国内,JBCD7E-03-Y1.i,2025-09-26 10:23:29,2099-01-01 00:00:00,NaN


In [20]:
df1.to_excel(fr"C:\Users\zhangbon\Desktop\长尾明细-魏总.xlsx", index=False)


In [24]:
df2 = pd.merge(df,df1[['物料编码','渠道','是否长尾型号']],on=['物料编码','渠道'],how='left')
df2['是否长尾型号'] = df2['是否长尾型号'].fillna('否')
df2

,发货类别,发货仓,渠道,大区/渠道A,收货片区/门店,收货城市,收货仓,是否三四级,承运方式,计划发货时间,...,周次,发货月份,物料编码,停止销售时间,停止发货时间,产品组,标准型号,产品型号,国内/海外,是否长尾型号
0,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,1,2024年1月,1001002400000,2025-06-19 09:20:56,NaT,吸油烟机,J1,CXW-258-J1(不带罩),国内,否
1,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,1,2024年1月,1002003700002,2026-01-04 12:12:01,NaT,灶具,02-HECB,JZT-02-HECB-12T,国内,否
2,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,1,2024年1月,1001000500376,2025-06-19 09:20:56,NaT,吸油烟机,JQ31A,CXW-358-JQ32A(不带罩）,国内,否
3,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,1,2024年1月,1001000500361,2024-08-30 15:22:25,2025-12-08 12:00:00,吸油烟机,JQ01TY,CXW-258-JQ01TY(不带罩),国内,否
4,调拨出库单,第一工业园立体仓,零售,NaN,无,南昌市,赣闽大区南昌库,NaN,干线tms,2024-01-02,...,1,2024年1月,1001001500097,2025-11-24 16:51:46,NaT,吸油烟机,Z7T,CXW-358-Z7T（不带罩）,国内,否
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
522703,调拨出库单,第二工业园立体仓,工程,NaN,无,深圳市,LTC惠州RDC工程仓,NaN,干线tms,2024-12-31,...,1,2024年12月,1001000800336,2022-07-07 12:00:00,NaT,吸油烟机,EH37,CXW-258-EH38,国内,否
522704,调拨出库单,第二工业园立体仓,零售,NaN,无,南京市,华东大区事业合作商-南京库,NaN,干线tms,2024-12-31,...,1,2024年12月,1003000500027,NaT,NaT,消毒柜,ZTD100F-J29,ZTD100F-J29,国内,否
522705,调拨出库单,第二工业园立体仓,零售,NaN,无,南京市,华东大区事业合作商-南京库,NaN,干线tms,2024-12-31,...,1,2024年12月,1003000500019,NaT,NaT,消毒柜,ZTD100J-J25S,ZTD100J-J25S,国内,否
522706,调拨出库单,第二工业园立体仓,零售,NaN,无,南京市,华东大区事业合作商-南京库,NaN,干线tms,2024-12-31,...,1,2024年12月,1009001100002,2025-08-01 13:38:39,NaT,蒸烤烹饪机,ZK50-01-F1.i,ZK50-01-F1.i,国内,否


In [25]:
df_result = pd.DataFrame()
df_result['产品类别'] = productgroup_map.keys()
df_result
for k,v in productgroup_map.items():
    vals1 = df2[(df2['产品组'].isin(v))&(df2['是否长尾型号']=='是')&(df2['渠道']=='零售')]['产品型号'].nunique()
    df_result.loc[df_result['产品类别']==k,'零售长尾产品型号数量'] = vals1
    vals2 = df2[(df2['产品组'].isin(v))&(df2['渠道']=='零售')]['产品型号'].nunique()
    df_result.loc[df_result['产品类别']==k,'零售产品型号数量'] = vals2
    df_result.loc[df_result['产品类别']==k,'零售长尾产品型号占比'] = vals1/vals2 if vals2!=0 else 0
    
    vals1 = df2[(df2['产品组'].isin(v))&(df2['是否长尾型号']=='是')&(df2['渠道']=='工程')]['产品型号'].nunique()
    df_result.loc[df_result['产品类别']==k,'工程长尾产品型号数量'] = vals1
    vals2 = df2[(df2['产品组'].isin(v))&(df2['渠道']=='工程')]['产品型号'].nunique()
    df_result.loc[df_result['产品类别']==k,'工程产品型号数量'] = vals2
    df_result.loc[df_result['产品类别']==k,'工程长尾产品型号占比'] = vals1/vals2 if vals2!=0 else 0
    
    vals1 = df2[(df2['产品组'].isin(v))&(df2['是否长尾型号']=='是')&(df2['渠道']=='电商')]['产品型号'].nunique()
    df_result.loc[df_result['产品类别']==k,'电商长尾产品型号数量'] = vals1
    vals2 = df2[(df2['产品组'].isin(v))&(df2['渠道']=='电商')]['产品型号'].nunique()
    df_result.loc[df_result['产品类别']==k,'电商产品型号数量'] = vals2 
    df_result.loc[df_result['产品类别']==k,'电商长尾产品型号占比'] = vals1/vals2 if vals2!=0 else 0
    

    vals1 = df2[(df2['产品组'].isin(v))&(df2['是否长尾型号']=='是')]['产品型号'].nunique()
    df_result.loc[df_result['产品类别']==k,'全渠道长尾产品型号数量'] = vals1
    vals2 = df2[(df2['产品组'].isin(v))]['产品型号'].nunique()
    df_result.loc[df_result['产品类别']==k,'全渠道产品型号数量'] = vals2
    df_result.loc[df_result['产品类别']==k,'全渠道长尾产品型号占比'] = vals1/vals2 if vals2!=0 else 0
    
df_result


,产品类别,零售长尾产品型号数量,零售产品型号数量,零售长尾产品型号占比,工程长尾产品型号数量,工程产品型号数量,工程长尾产品型号占比,电商长尾产品型号数量,电商产品型号数量,电商长尾产品型号占比,全渠道长尾产品型号数量,全渠道产品型号数量,全渠道长尾产品型号占比
0,吸油烟机,24.0,96.0,0.250000,19.0,115.0,0.165217,8.0,110.0,0.072727,46.0,222.0,0.207207
1,灶具,35.0,196.0,0.178571,4.0,101.0,0.039604,2.0,120.0,0.016667,39.0,319.0,0.122257
2,蒸烤微合计,12.0,54.0,0.222222,2.0,38.0,0.052632,2.0,68.0,0.029412,12.0,89.0,0.134831
3,灶集成,7.0,51.0,0.137255,0.0,12.0,0.000000,8.0,33.0,0.242424,12.0,56.0,0.214286
4,消毒柜,3.0,36.0,0.083333,3.0,27.0,0.111111,0.0,28.0,0.000000,6.0,55.0,0.109091
5,热水器,6.0,76.0,0.078947,1.0,23.0,0.043478,0.0,36.0,0.000000,7.0,100.0,0.070000
6,净水机,0.0,33.0,0.000000,0.0,15.0,0.000000,0.0,27.0,0.000000,0.0,36.0,0.000000
7,洗碗机,13.0,70.0,0.185714,6.0,52.0,0.115385,1.0,102.0,0.009804,15.0,137.0,0.109489


In [23]:
with pd.ExcelWriter(fr'D:\000物料报表\{month_date}\单型号贡献-低效-长尾\长尾统计结果-26新版-魏总.xlsx') as writer:
    df_result.to_excel(writer,sheet_name='分渠道产品型号统计',index=False)
